## Reading Data from an API

A web API is an API over the web.

Think of an API like a restaurant's menu: you don't need to know how the kitchen works internally, you just pick an item from a fixed menu, and the kitchen sends back what you asked for. 

A web API works the same way: you send a request to a fixed endpoint, and it sends back data, usually in a clean, structured format like JSON.

In this lab, we'll pull real currency exchange rate data from a free public API and turn it into a pandas DataFrame we can explore.

### Setting up

We'll use Python's built-in `urllib` to make the request (no extra installation needed) and `json` to parse the response.

In [19]:
import urllib.request
import urllib.error
import json

### About the Frankfurter API

Frankfurter is a free, open-source currency exchange rate API, sourced from the European Central Bank (and other central banks). It's well-established, historical data goes back to **1948** and covers **201 currencies**.

**Endpoints:**
- **Latest rates:** `/v1/latest?base=USD`
- **Historical (specific date):** `/v1/1999-01-04?base=USD&symbols=EUR`
- **Time series (date range):** `/v1/2010-01-01..2010-01-31`
- **List of currencies:** `/v1/currencies`: returns currency codes along with their full names (e.g. `"EUR": "Euro"`)


### Making our request

Let's fetch the latest exchange rates, using the US Dollar as our base currency.

In [24]:
url = "https://api.frankfurter.dev/v1/latest?base=USD"

#This is the address of the API endpoint we're calling. 
#?base=USD: This is a query parameter.

# Some APIs block requests that don't look like they're coming from a browser,
# so we set a User-Agent header

req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

try:
    response = urllib.request.urlopen(req) #urlopen(req) returns a response object, and .read() pulls the raw content out of it
    data = response.read()
    print("Request successful!")

except urllib.error.HTTPError as e:
    print(f"HTTP Error: {e.code} - {e.reason}") 
    
except urllib.error.URLError as e:
    print(f"URL Error: {e.reason}")

Request successful!


Two separate except blocks, catching two different failure types:

- HTTPError: The request reached the server just fine, but the server responded with an error (like 404 Not Found, or the 403 Forbidden).
- URLError: The request never even reached the server at all (e.g. no internet connection, or a typo in the URL/domain that doesn't resolve).

In [25]:
data

b'{"amount":1.0,"base":"USD","date":"2026-08-24","rates":{"AUD":1.3963,"BRL":5.1499,"CAD":1.3849,"CHF":0.80264,"CNY":6.7227,"CZK":20.662,"DKK":6.4092,"EUR":0.85734,"GBP":0.73345,"HKD":7.8368,"HUF":310.83,"IDR":17713,"ILS":2.9922,"INR":95.76,"ISK":121.06,"JPY":159.12,"KRW":1384.26,"MXN":16.9282,"MYR":4.0425,"NOK":9.3146,"NZD":1.6776,"PHP":61.719,"PLN":3.6932,"RON":4.5014,"SEK":9.4997,"SGD":1.2705,"THB":32.685,"TRY":48.081,"ZAR":16.0029}}'

data is just a string of characters that happens to look dictionary-shaped. `json.loads()` is the function that actually converts that text into a real Python object: a dictionary

In [26]:
rates_data = json.loads(data)
type(rates_data)

dict

`rates_data` is a dictionary with four keys: `amount`, `base`, `date`, and `rates`, where `rates` is itself a nested dictionary mapping currency codes to exchange rates.

In [27]:
rates_data

{'amount': 1.0,
 'base': 'USD',
 'date': '2026-08-24',
 'rates': {'AUD': 1.3963,
  'BRL': 5.1499,
  'CAD': 1.3849,
  'CHF': 0.80264,
  'CNY': 6.7227,
  'CZK': 20.662,
  'DKK': 6.4092,
  'EUR': 0.85734,
  'GBP': 0.73345,
  'HKD': 7.8368,
  'HUF': 310.83,
  'IDR': 17713,
  'ILS': 2.9922,
  'INR': 95.76,
  'ISK': 121.06,
  'JPY': 159.12,
  'KRW': 1384.26,
  'MXN': 16.9282,
  'MYR': 4.0425,
  'NOK': 9.3146,
  'NZD': 1.6776,
  'PHP': 61.719,
  'PLN': 3.6932,
  'RON': 4.5014,
  'SEK': 9.4997,
  'SGD': 1.2705,
  'THB': 32.685,
  'TRY': 48.081,
  'ZAR': 16.0029}}

In [28]:
print(rates_data.keys())
print(f"Base currency: {rates_data['base']}")
print(f"Date: {rates_data['date']}")

dict_keys(['amount', 'base', 'date', 'rates'])
Base currency: USD
Date: 2026-08-24


In [29]:
# The actual exchange rates are nested one level deeper
rates_data['rates']

{'AUD': 1.3963,
 'BRL': 5.1499,
 'CAD': 1.3849,
 'CHF': 0.80264,
 'CNY': 6.7227,
 'CZK': 20.662,
 'DKK': 6.4092,
 'EUR': 0.85734,
 'GBP': 0.73345,
 'HKD': 7.8368,
 'HUF': 310.83,
 'IDR': 17713,
 'ILS': 2.9922,
 'INR': 95.76,
 'ISK': 121.06,
 'JPY': 159.12,
 'KRW': 1384.26,
 'MXN': 16.9282,
 'MYR': 4.0425,
 'NOK': 9.3146,
 'NZD': 1.6776,
 'PHP': 61.719,
 'PLN': 3.6932,
 'RON': 4.5014,
 'SEK': 9.4997,
 'SGD': 1.2705,
 'THB': 32.685,
 'TRY': 48.081,
 'ZAR': 16.0029}

### From dictionary to DataFrame

Now let's turn the rates dictionary into a proper pandas DataFrame, one row per currency, with its code and exchange rate.

In [31]:
import pandas as pd

rates_dict = rates_data['rates']
rates_dict


{'AUD': 1.3963,
 'BRL': 5.1499,
 'CAD': 1.3849,
 'CHF': 0.80264,
 'CNY': 6.7227,
 'CZK': 20.662,
 'DKK': 6.4092,
 'EUR': 0.85734,
 'GBP': 0.73345,
 'HKD': 7.8368,
 'HUF': 310.83,
 'IDR': 17713,
 'ILS': 2.9922,
 'INR': 95.76,
 'ISK': 121.06,
 'JPY': 159.12,
 'KRW': 1384.26,
 'MXN': 16.9282,
 'MYR': 4.0425,
 'NOK': 9.3146,
 'NZD': 1.6776,
 'PHP': 61.719,
 'PLN': 3.6932,
 'RON': 4.5014,
 'SEK': 9.4997,
 'SGD': 1.2705,
 'THB': 32.685,
 'TRY': 48.081,
 'ZAR': 16.0029}

In [32]:
a = list(rates_dict.items())
a

[('AUD', 1.3963),
 ('BRL', 5.1499),
 ('CAD', 1.3849),
 ('CHF', 0.80264),
 ('CNY', 6.7227),
 ('CZK', 20.662),
 ('DKK', 6.4092),
 ('EUR', 0.85734),
 ('GBP', 0.73345),
 ('HKD', 7.8368),
 ('HUF', 310.83),
 ('IDR', 17713),
 ('ILS', 2.9922),
 ('INR', 95.76),
 ('ISK', 121.06),
 ('JPY', 159.12),
 ('KRW', 1384.26),
 ('MXN', 16.9282),
 ('MYR', 4.0425),
 ('NOK', 9.3146),
 ('NZD', 1.6776),
 ('PHP', 61.719),
 ('PLN', 3.6932),
 ('RON', 4.5014),
 ('SEK', 9.4997),
 ('SGD', 1.2705),
 ('THB', 32.685),
 ('TRY', 48.081),
 ('ZAR', 16.0029)]

In [34]:
rates_df = pd.DataFrame(a, columns=['Currency', 'Rate'])
rates_df.head()

,Currency,Rate
0,AUD,1.39630
1,BRL,5.14990
2,CAD,1.38490
3,CHF,0.80264
4,CNY,6.72270


In [35]:
rates_df.shape

(29, 2)